# Fetch MCP server

This notebook deploys the upstream [GoFetch](https://github.com/StacklokLabs/gofetch) MCP server and uses its `fetch` tool to retrieve a web page as Markdown. See the [README](./README.md) for terminal and client configuration instructions.

MCP servers must be enabled on your prokube cluster (v1.9.0 or later).

## Setup

Load this repository's helpers from [mcp.py](../../src/pk_helpers/mcp.py) and [api_key.py](../../src/pk_helpers/api_key.py). `%run -n` loads their functions and classes into the notebook namespace without running guarded CLI code. `McpSession` is a minimal MCP client; any MCP client library can use the same URLs.

In [ ]:
# mcp.py supplies workspace_mcp_url, server_mcp_url, connect_when_ready and McpSession.
%run -n ../../src/pk_helpers/mcp.py
# api_key.py supplies get_or_create_mcp_api_key, which reads MCP_API_KEY or prompts.
%run -n ../../src/pk_helpers/api_key.py

## Deploy

In [ ]:
!kubectl apply -f fetch-server.yaml
!kubectl wait --for=condition=Ready mcpservers.toolhive.stacklok.dev/fetch-example --timeout=3m

## Connect from this Lab

Labs use the in-cluster Agent Gateway host to reach `/_platform/mcp/<workspace>` without an API key. This workspace federation combines all MCP servers and prefixes each tool with its server name, for example `fetch-example_fetch`.

In [ ]:
with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
    namespace = f.read().strip()

internal_url = workspace_mcp_url(namespace)
print(internal_url)

In [ ]:
session = connect_when_ready(internal_url, tool_prefix="fetch-example_")
session.list_tools()

Fetch a web page:

In [ ]:
result = session.call_tool("fetch-example_fetch", {"url": "https://example.com"})
print(result["content"][0]["text"])

## Connect with an API key

Clients outside the cluster use the server's own route, `https://<your-prokube-domain>/svc/mcp/<workspace>/fetch-example`, with an API key. Create a key with access to `fetch-example` on the [API Keys page](/pkui/ai-gateway/keys) (`https://<your-prokube-domain>/pkui/ai-gateway/keys`), then paste it when prompted below.

The notebook uses the in-cluster host to exercise this same single-server `/svc/mcp` authentication route. Preview its URL before retrieving the key and connecting. Tool names on this route are unprefixed.

In [ ]:
api_key_url = server_mcp_url(namespace, "fetch-example")
print(api_key_url)

In [ ]:
api_key = get_or_create_mcp_api_key()
server_session = connect_when_ready(api_key_url, api_key=api_key)
server_session.list_tools()

## Clean up

In [ ]:
!kubectl delete -f fetch-server.yaml